# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashiba713/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings and methodology questions

#### Finding 1 — Content lifecycle

The research paper reports that growing pages averaged about **185 days old**, while declining pages averaged about **228 days old**.

My methodology question is: **how exactly is the growing versus declining label constructed, and does the validation or comparison design prevent information from the same underlying entity from influencing both sides?**

This matters because page age may be associated with the observed outcome, but the result should be interpreted according to how the labels and comparison groups were constructed.

#### Finding 2 — Growth prediction

The paper reports approximately **90% accuracy on unseen pages from the same brands** and approximately **75% accuracy on unseen brands**.

My methodology question is: **does the validation design fully separate unseen brands from the training data, and are these accuracy figures calculated only from held-out predictions?**

The distinction between same-brand and unseen-brand performance is useful because it helps show how much the model's performance may depend on familiarity with a brand. I would want the validation design to match the generalization claim being made.

These are methodology questions rather than conclusions that the findings are incorrect. They are the same kinds of questions I should apply to my own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation: before and after

My Week-5 model is intended to support review prioritization across clients.

A random row split can place pages from the same client in both training and test data. This can make performance look stronger because the model sees related examples from the same client during training.

As an honest alternative, I use **GroupKFold by `client_id`**. Every row belonging to a client stays in either the training or test portion of a fold.

I report the random-split result alongside the grouped result because the gap itself is informative. A large drop would suggest that random validation was benefiting from within-client similarity.

The main metric is **Precision@50**, matching the Week-5 decision depth.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42

# Load the same dataset used in ML-08.
df = pd.read_csv(
    "https://raw.githubusercontent.com/ashiba713/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
)

# Same observed target as ML-08.
df["target_decline"] = (
    df["trend_direction"] == "down"
).astype(int)

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_with_impressions",
    "days_with_sessions"
]

X = df[feature_columns].copy()
y = df["target_decline"].copy()
groups = df["client_id"].copy()

print("Rows:", len(df))
print("Clients:", groups.nunique())
print("Base rate:", round(y.mean(), 4))

Rows: 30000
Clients: 32
Base rate: 0.5421


In [2]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    order = np.argsort(
        -scores,
        kind="mergesort"
    )

    return float(
        y_true[order[:k]].mean()
    )


def make_logistic_model():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED
                )
            )
        ]
    )

In [3]:
# -----------------------------
# BEFORE: random row split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

random_model = make_logistic_model()

random_model.fit(
    X_train,
    y_train
)

random_scores = random_model.predict_proba(
    X_test
)[:, 1]

random_p50 = precision_at_k(
    y_test,
    random_scores,
    50
)

print("Random split Precision@50:",
      round(random_p50, 4))

print("Random split base rate:",
      round(y_test.mean(), 4))

Random split Precision@50: 0.84
Random split base rate: 0.542


In [4]:
# -----------------------------
# AFTER: grouped validation
# -----------------------------

gkf = GroupKFold(
    n_splits=5
)

grouped_rows = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        np.zeros(len(y)),
        y,
        groups=groups
    ),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    train_groups = groups.iloc[train_idx]
    test_groups = groups.iloc[test_idx]

    overlap = set(train_groups).intersection(
        set(test_groups)
    )

    assert len(overlap) == 0

    model = make_logistic_model()

    model.fit(
        X_train,
        y_train
    )

    scores = model.predict_proba(
        X_test
    )[:, 1]

    p50 = precision_at_k(
        y_test,
        scores,
        50
    )

    grouped_rows.append(
        {
            "fold": fold,
            "precision@50": p50,
            "base_rate": y_test.mean(),
            "n_test": len(test_idx)
        }
    )

grouped_results = pd.DataFrame(
    grouped_rows
)

print(grouped_results.round(4))

grouped_p50 = (
    grouped_results["precision@50"]
    .mean()
)

print(
    "\nGrouped mean Precision@50:",
    round(grouped_p50, 4)
)

   fold  precision@50  base_rate  n_test
0     1          0.70     0.4902    7008
1     2          0.64     0.6454    5731
2     3          0.80     0.3795    5753
3     4          0.66     0.6222    5755
4     5          0.90     0.5847    5753

Grouped mean Precision@50: 0.74


In [5]:
before_after = pd.DataFrame(
    {
        "validation": [
            "Random row split",
            "Grouped by client"
        ],
        "precision@50": [
            random_p50,
            grouped_p50
        ],
        "base_rate": [
            y_test.mean(),
            grouped_results["base_rate"].mean()
        ]
    }
)

before_after.round(4)

,validation,precision@50,base_rate
0,Random row split,0.84,0.5847
1,Grouped by client,0.74,0.5444


### Interpretation

The random split and grouped split answer different questions.

The random split estimates performance when related pages from the same clients can appear in both training and testing.

The grouped split is more conservative for my intended use because the model is evaluated on clients that were not present in its training fold.

I therefore treat the grouped result as the more honest estimate for cross-client decision support. Any difference between the random and grouped results is reported as a validation finding rather than hidden.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

I checked the final Week-5 feature set against the main leakage categories:

1. **Label-derived features:** `trend_direction`, `trend_pct`, and `target_decline` are excluded from model inputs.
2. **Known leakage signal:** `impression_change_pct` is excluded because the earlier audit showed that it reproduced the observed trend calculation.
3. **Future or overlapping windows:** features that directly encode the outcome window are excluded from the modeling feature set.
4. **Decision-derived features:** existing system scores or flags are not used as model inputs.
5. **Identifiers:** `client_id` is used only to create grouped validation folds and is not a model feature.

The remaining model features are treated as candidate signals rather than causal variables. Their usefulness must be judged using held-out predictions.

In [6]:
# Explicit feature leakage checks

suspect_columns = [
    "trend_direction",
    "trend_pct",
    "target_decline",
    "impression_change_pct",
    "client_id",
    "content_id"
]

leaks_in_features = [
    col
    for col in suspect_columns
    if col in feature_columns
]

print("Suspect columns found in model features:")
print(leaks_in_features)

assert leaks_in_features == []

print(
    "\nLeakage check passed: "
    "no identified label/ID leakage columns are model features."
)

Suspect columns found in model features:
[]

Leakage check passed: no identified label/ID leakage columns are model features.


In [7]:
# Deliberate leakage attack.
#
# If the evaluation is working correctly, adding a direct copy
# of the observed label should make the score extremely high.
# We then remove it and keep the honest feature set.

X_leaky = X.copy()

X_leaky["DELIBERATE_LEAK"] = y.values

leaky_model = make_logistic_model()

leaky_model.fit(
    X_leaky,
    y
)

leaky_scores = leaky_model.predict_proba(
    X_leaky
)[:, 1]

leaky_p50 = precision_at_k(
    y,
    leaky_scores,
    50
)

print(
    "Deliberate-leak Precision@50:",
    round(leaky_p50, 4)
)

print(
    "\nThe deliberate leak is used only as a diagnostic."
)

Deliberate-leak Precision@50: 1.0

The deliberate leak is used only as a diagnostic.


### Leakage conclusion

The deliberate leakage test is a diagnostic rather than a valid model.

The final Week-5 feature set does not contain the observed target, its direct label-derived columns, the previously rejected impression-change leakage feature, or client/content identifiers as predictive inputs.

The model's reported performance should therefore come from the clean feature set and held-out validation rather than from the deliberate leak.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

### Original bold claim

> "The model can predict which pages will decline and identify the pages that should be refreshed first."

### Evidence-safe rewrite

> **"On the evaluated anonymized dataset, the model produced a measured ranking of pages by estimated probability of the observed declining label. Under grouped-by-client validation, this provides directional decision support for prioritizing pages for human review. It does not establish that the model will predict future decline reliably in deployment, that refreshing a page will cause improvement, or that the model predicts Google's ranking behavior."**

The revised claim is narrower because the evidence comes from an anonymized dataset and an observed historical label. The result is therefore treated as **measured and directional decision support**, not as a causal or deployment guarantee.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.